In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset
import numpy as np

In [ ]:
# Load datasets CONLL-2003
dataset = load_dataset("conll2003")
train_data = dataset["train"]
test_data = dataset["test"]

tags = train_data.features["ner_tags"].feature.names
num_tags = len(tags)

In [ ]:
# Process data
def preprocess_data(data):
    texts = []
    labels = []
    for item in data:
        texts.append(item["tokens"])
        labels.apeend(item["ner_tags"])

    word_set = set()
    for text in texts:
        word_set.update(text)
    word2idx = {word: i+1 for i, word in enumerate(word_set)}
    word2idx["<PAD>"] = 0

    text_seqs = [[word2idx[word] for word in text] for text in texts]

    max_len = 50
    text_seqs_pad = pad_sequences(text_seqs, maxlen=max_len, padding="post")
    labels_pad = pad_sequences(labels, maxlen=max_len, padding="post")
    
    return text_seqs_pad, labels_pad, word2idx

x_train, y_train, word2idx = preprocess_data(train_data)
x_test, y_test, _ = preprocess_data(test_data)
vocab_size = len(word2idx)

In [ ]:
# Configure binary GRU
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=50),
    Bidirectional(GRU(128, return_sequences=True)),
    Dense(num_tags, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crosssentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Train model
model.fit(
    x_train, y_train.reshape(*y_train.shape, 1),
    batch_size=32,
    epochs=3,
    validation_split=0.2
)

In [ ]:
# Evaluate model
test_loss, test_acc = model.evaluate(x_test, y_test.reshape(*y_train.shape, 1))
print(f"测试准确率：{test_acc:.4f}")

In [ ]:
# Predict test
def predict_ner(text):
    tokens = text.split()
    text_seq = [word2idx.get(word, 0) for word in tokens]
    text_pad = pad_sequences([text_seq], maxlen=50, padding="post")

    pred = model.predict(text_pad)[0]
    pred_tags = [tags[np.argmax(token_pred)] for token_pred in pred[:len(tokens)]]

    return list(zip(tokens, pred_tags))

# test
print(predict_ner("Barack Obama was born in Hawaii."))